In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings

In [13]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [14]:
warnings.filterwarnings("ignore", category=FutureWarning)

In [18]:
fashion_mnist = fetch_openml(
"fashion-MNIST",
version=1,
as_frame=False
)

X = fashion_mnist.data
y = fashion_mnist.target.astype(int)

print(f"Total dataset size: {X.shape[0]} images, each with {X.shape[1]} pixels.")

Total dataset size: 70000 images, each with 784 pixels.


In [20]:
X_subset, _, y_subset, _ = train_test_split(
X,
y,
train_size=12000,
stratify=y,
random_state=42
)

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
X_subset,
y_subset,
test_size=2000,
stratify=y_subset,
random_state=42
)

print(f"Training images: {X_train.shape[0]}")
print(f"Testing images: {X_test.shape[0]}")

Training images: 10000
Testing images: 2000


In [22]:
print(f"Before scaling -> Min: {X_train.min()}, Max: {X_train.max()}")

Before scaling -> Min: 0, Max: 255


In [23]:

X_train = X_train / 255.0
X_test = X_test / 255.0

print(
f"After scaling -> Min: {X_train.min()}, "
f"Max: {X_train.max()}"
)


After scaling -> Min: 0.0, Max: 1.0


In [27]:
k_values = [1, 3, 5, 7, 9, 15]
results = {}

print(
f"{'K value':<8} | "
f"{'Accuracy':<10} | "
f"{'Prediction time (seconds)':<25}"
)

print("-" * 50)


for k in k_values:
    knn = KNeighborsClassifier(
    n_neighbors=k,
    metric="euclidean",
    n_jobs=-1
)
knn.fit(X_train, y_train)

# Measure prediction time
start_time = time.time()

y_pred = knn.predict(X_test)

prediction_time = time.time() - start_time

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

# Store results
results[k] = {
"accuracy": accuracy,
"prediction_time": prediction_time
}

# Display results
print(
f"{k:<8} | "
f"{accuracy * 100:<9.2f}% | "
f"{prediction_time:<25.2f}"
)


# --------------------------------------------------
# 5. Find Best K
# --------------------------------------------------

best_k = max(
results,
key=lambda k: results[k]["accuracy"]
)

print("\nBest K value:", best_k)
print(
f"Best Accuracy: "
f"{results[best_k]['accuracy'] * 100:.2f}%"
)

K value  | Accuracy   | Prediction time (seconds)
--------------------------------------------------
15       | 80.25    % | 0.21                     

Best K value: 15
Best Accuracy: 80.25%


In [28]:
best_knn = KNeighborsClassifier(
n_neighbors=best_k,
metric="euclidean",
n_jobs=-1
)

best_knn.fit(X_train, y_train)

y_pred = best_knn.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.85      0.80       200
           1       0.97      0.95      0.96       200
           2       0.66      0.69      0.67       200
           3       0.87      0.82      0.85       200
           4       0.71      0.68      0.70       200
           5       0.99      0.70      0.82       200
           6       0.55      0.54      0.55       200
           7       0.78      0.91      0.84       200
           8       0.96      0.93      0.94       200
           9       0.84      0.94      0.89       200

    accuracy                           0.80      2000
   macro avg       0.81      0.80      0.80      2000
weighted avg       0.81      0.80      0.80      2000



In [29]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Confusion Matrix:
[[170   0   1   6   2   0  18   0   3   0]
 [  1 190   5   3   1   0   0   0   0   0]
 [  3   0 138   1  29   0  28   0   1   0]
 [ 10   4   3 165   6   0  12   0   0   0]
 [  2   1  29  11 136   0  21   0   0   0]
 [  1   0   0   0   0 141   2  37   1  18]
 [ 40   0  29   4  16   0 108   0   3   0]
 [  0   0   0   0   0   1   0 182   0  17]
 [  0   0   4   0   1   0   5   4 186   0]
 [  0   0   0   0   0   0   1  10   0 189]]
